# Chapter 6 &mdash; DeMorgan's Law for DFA, Verified by Isomorphism

**Concept 12 of the Chapter 6 decomposition:** *DeMorgan's Law for DFA, Verified by Isomorphism*

$L_1\cap L_2 = \overline{\overline{L_1}\cup\overline{L_2}}$ &mdash; confirmed by minimizing both sides.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6-DFAOps/Concept-DeMorgan-For-DFA/Concept-DeMorgan-For-DFA.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.AnimateDFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateDFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateDFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


DeMorgan's law holds for languages:
$$L_1 \cap L_2 = \overline{\overline{L_1} \cup \overline{L_2}}.$$

So intersection is **redundant** as a primitive: complement and union suffice. This is
the same closure argument you will use for regular expressions in Chapter 10.

And it is **checkable**. Build both sides, minimize both, and ask `iso_dfa`. By
Myhill&ndash;Nerode a `True` answer is a proof for these particular languages &mdash; the
strongest machine-checked evidence available without a general proof.

## 2. Definitions

### Two languages

In [ ]:
even0 = md2mc('''DFA
IF : 0 -> Od
IF : 1 -> IF
Od : 0 -> IF
Od : 1 -> Od
''')
even1 = md2mc('''DFA
IF : 1 -> Od
IF : 0 -> IF
Od : 1 -> IF
Od : 0 -> Od
''')

### Both sides of DeMorgan's law

In [ ]:
def lhs(A, B): return min_dfa(pruneUnreach(intersect_dfa(A, B)))
def rhs(A, B): return min_dfa(pruneUnreach(
                   comp_dfa(union_dfa(comp_dfa(A), comp_dfa(B)))))

<!-- nav-strip -->

---

&larr;&nbsp;[Ch6&nbsp;11.&nbsp;Worked Example: Union, Minimization, and the Two Comparison Predicates](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6-DFAOps/Concept-Worked-Union-Minimization/Concept-Worked-Union-Minimization.ipynb) &nbsp;&middot;&nbsp; [**Chapter 6** index](https://github.com/ganeshutah/Jove/blob/master/Chapter6-DFAOps/README.md) &nbsp;&middot;&nbsp; [Ch7&nbsp;1.&nbsp;Nondeterminism as Forking Tokens, and as Guessing](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter7-NFA/Concept-Forking-Tokens/Concept-Forking-Tokens.ipynb)&nbsp;&rarr;

---

## 3. Tests

The two sides minimize to the same size.

In [ ]:
L, R = lhs(even0, even1), rhs(even0, even1)
print("LHS (intersection)     : %d states" % len(L["Q"]))
print("RHS (DeMorgan version) : %d states" % len(R["Q"]))
assert len(L["Q"]) == len(R["Q"])

And they are **isomorphic** &mdash; which by Myhill&ndash;Nerode means identical languages.

In [ ]:
print("langeq_dfa :", langeq_dfa(L, R))
print("iso_dfa    :", iso_dfa(L, R))
assert langeq_dfa(L, R) and iso_dfa(L, R)
print("\nMinimal + isomorphic  =>  the same language, proved by the theorem.")

Cross-checked against the arithmetic specification, both ways.

In [ ]:
from itertools import product
strs = [''.join(p) for k in range(11) for p in product('01', repeat=k)]
spec = lambda s: s.count('0') % 2 == 0 and s.count('1') % 2 == 0
for name, X in [('LHS', L), ('RHS', R)]:
    assert all(accepts_dfa(X, s) == spec(s) for s in strs)
    print("%s matches the spec on all %d strings" % (name, len(strs)))

The dual law too: $L_1\cup L_2 = \overline{\overline{L_1}\cap\overline{L_2}}$.

In [ ]:
U1 = min_dfa(pruneUnreach(union_dfa(even0, even1)))
U2 = min_dfa(pruneUnreach(comp_dfa(intersect_dfa(comp_dfa(even0), comp_dfa(even1)))))
print("dual law holds? langeq=%s iso=%s" % (langeq_dfa(U1, U2), iso_dfa(U1, U2)))
assert langeq_dfa(U1, U2) and iso_dfa(U1, U2)

So intersection is redundant &mdash; complement and union generate it.

In [ ]:
print("primitive set {comp, union} suffices for the Boolean operations on DFA.")
print("Chapter 10 makes the same argument for regular expressions.")

## 4. Animation

Both sides of DeMorgan's law produce this one minimal machine.

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(lhs(even0, even1), FuseEdges=True)

## 5. Exercises


1. Verify DeMorgan's law on two machines of different sizes.
2. Is `iso_dfa` on minimized machines a **proof**, or only strong evidence? Justify.
3. Which other operations are redundant given complement and union?

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for all 254 concepts.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter6-DFAOps/Concept-DeMorgan-For-DFA')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')